# D3 Aaya - Online/Adaptive Retrieval inside GraphRAG

**Owner:** Aaya  
**Main evidence:** compare static, topic-gated, and feedback-adaptive retrieval inside the D3 GraphRAG pipeline.

This notebook is fixed to be a clear D3 working scaffold, not a broken TODO-only notebook. It does **not** fabricate final GraphRAG numbers. Instead, it loads the D2 online-adaptation results as baseline evidence, defines the required D3 output schema, and shows exactly what must be filled after the GraphRAG executor and PEFT/QLoRA final scope are available.

## 1. What Aaya owns in D3

Aaya's D3 job is to answer this question:

> Does the online learner improve GraphRAG evidence/answer quality compared with a fixed static retrieval route?

The comparison should include:

- `static`: fixed hybrid retrieval weights.
- `topic_gated`: River predicts topic, then fixed topic retrieval profile is used.
- `adaptive`: River predicts topic, then FeedbackAdapter changes retrieval weights from feedback.

Required final output file:

```text
reports/tables/d3_online_retrieval_comparison.csv
```

Related files:

```text
notebooks/D3_04_Aaya_online_graphrag_adaptation.ipynb
notebooks/D3_graphrag_eval_safety.ipynb
src/learning/feedback_adapter.py
src/learning/river_topic_classifier.py
reports/member_sections/aaya_d3_adaptation_section.md
```

In [ ]:
# 2. Robust setup
# This cell works whether the notebook is opened from the repo root or from notebooks/.
from pathlib import Path
import json
import math
import time
import pandas as pd

candidate_roots = []
cwd = Path.cwd().resolve()
candidate_roots.extend([cwd, cwd.parent])

# Local absolute fallback for this project machine.
candidate_roots.append(Path(r"D:\BUID\Year 4 2025 - 2026\third semester\Special Topics in AI\Project\climate_evidence_graphrag_agent"))

PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / "src").exists() and (candidate / "notebooks").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate project root. Open this notebook from the repo root or notebooks folder.")

REPORTS_TABLES = PROJECT_ROOT / "reports" / "tables"
REPORTS_TABLES.mkdir(parents=True, exist_ok=True)

D2_ONLINE_SUMMARY = REPORTS_TABLES / "d2_online_adaptation_summary.csv"
D2_ONLINE_VS_STATIC = REPORTS_TABLES / "d2_online_vs_static.csv"
D3_OUTPUT = REPORTS_TABLES / "d3_online_retrieval_comparison.csv"
D3_TEMPLATE_OUTPUT = REPORTS_TABLES / "d3_online_retrieval_comparison_TEMPLATE.csv"

print("Project root:", PROJECT_ROOT)
print("D2 online summary exists:", D2_ONLINE_SUMMARY.exists())
print("D2 online vs static exists:", D2_ONLINE_VS_STATIC.exists())
print("D3 final output expected:", D3_OUTPUT)

## 3. Carry forward D2 correctly

D2 already tested online adaptation at the retrieval level. D3 should **not** simply copy D2. D3 should test whether online adaptation changes:

- graph-supported retrieval quality,
- answer faithfulness,
- answer relevance,
- citation correctness,
- latency,
- and behavior under simulated drift or feedback.

The D2 table is still useful because it provides the baseline starting point and proves that the River/FeedbackAdapter route already works.

In [ ]:
# 4. Load D2 online/adaptive metrics as baseline evidence.
# These are not final D3 GraphRAG metrics; they are carry-forward baseline evidence.

if D2_ONLINE_SUMMARY.exists():
    d2_summary = pd.read_csv(D2_ONLINE_SUMMARY)
elif D2_ONLINE_VS_STATIC.exists():
    d2_summary = pd.read_csv(D2_ONLINE_VS_STATIC)
else:
    d2_summary = pd.DataFrame()

if d2_summary.empty:
    print("D2 online metrics were not found. Aaya must run D2_04_Aaya_online_retrieval_adaptation.ipynb first.")
else:
    display(d2_summary)
    print("Loaded D2 baseline rows:", len(d2_summary))
    print("Columns:", list(d2_summary.columns))

In [ ]:
# 5. Define the required D3 schema for Aaya's final comparison.
# The final table must use this schema so the shared notebook and report can read it reliably.

D3_EXPECTED_COLUMNS = [
    "method",
    "phase",
    "prequential_topic_accuracy",
    "recall@5",
    "ndcg@5",
    "mrr@5",
    "faithfulness",
    "answer_relevance",
    "citation_hit_rate",
    "citation_correctness",
    "mean_latency_ms",
    "p95_latency_ms",
    "final_bm25_weight",
    "final_dense_weight",
    "helps_count",
    "hurts_count",
    "peft_qlora_mode",
    "graph_mode",
    "evaluation_status",
    "limitation",
]

schema_df = pd.DataFrame({
    "column": D3_EXPECTED_COLUMNS,
    "meaning": [
        "static, topic_gated, adaptive, or adaptive_peft if tuned model is used",
        "stable_pre_drift, concept_drift, stable_post_drift, or full_stream",
        "River topic accuracy measured predict-then-learn",
        "retrieval evidence recall at top 5",
        "ranking quality at top 5",
        "rank of first relevant evidence in top 5",
        "whether answer claims are supported by retrieved/cited evidence",
        "whether answer addresses the question",
        "whether retrieved/cited result matches expected evidence/topic",
        "whether citation document/page is verified",
        "average runtime",
        "95th percentile runtime",
        "final or average BM25 weight",
        "final or average dense weight",
        "number of queries where adaptive improved over static",
        "number of queries where adaptive hurt compared with static",
        "zero_shot, peft_qlora_tuned, or not_run",
        "vector_only, hybrid, graph_guided, or graph_guided_hybrid",
        "final, partial, or pending_graphrag_executor",
        "honest limitation for this row",
    ]
})

display(schema_df)

In [ ]:
# 6. Build a D3 planning table from D2 metrics without pretending it is final D3 evidence.
# This table helps Aaya see what still needs to be measured after GraphRAG is connected.

def safe_get(row, *names, default=pd.NA):
    for name in names:
        if name in row and pd.notna(row[name]):
            return row[name]
    return default

planning_rows = []
if not d2_summary.empty:
    for _, row in d2_summary.iterrows():
        method = str(row.get("method", "unknown"))
        planning_rows.append({
            "method": method,
            "phase": "full_stream_from_D2_baseline",
            "prequential_topic_accuracy": safe_get(row, "Prequential topic accuracy"),
            "recall@5": safe_get(row, "Recall@5"),
            "ndcg@5": safe_get(row, "nDCG@5"),
            "mrr@5": safe_get(row, "MRR@5"),
            "faithfulness": pd.NA,
            "answer_relevance": pd.NA,
            "citation_hit_rate": safe_get(row, "Answer citation hit rate"),
            "citation_correctness": pd.NA,
            "mean_latency_ms": safe_get(row, "Mean latency ms"),
            "p95_latency_ms": pd.NA,
            "final_bm25_weight": safe_get(row, "Final mean BM25 weight"),
            "final_dense_weight": safe_get(row, "Final mean dense weight"),
            "helps_count": pd.NA,
            "hurts_count": pd.NA,
            "peft_qlora_mode": "not_run_yet",
            "graph_mode": "not_connected_yet",
            "evaluation_status": "D2_BASELINE_ONLY_NOT_FINAL_D3",
            "limitation": "GraphRAG executor, answer generation, citation verifier, and PEFT/QLoRA comparison still need to be run for final D3.",
        })

planning_df = pd.DataFrame(planning_rows, columns=D3_EXPECTED_COLUMNS)
if planning_df.empty:
    print("No D2 baseline rows available yet.")
else:
    display(planning_df)

In [ ]:
# 7. Optional: write a TEMPLATE file only, not the final D3 evidence file.
# Keep this False unless you intentionally want a template CSV for coordination.

WRITE_TEMPLATE_CSV = False

if WRITE_TEMPLATE_CSV and not planning_df.empty:
    planning_df.to_csv(D3_TEMPLATE_OUTPUT, index=False)
    print("Wrote template file:", D3_TEMPLATE_OUTPUT)
else:
    print("Template CSV not written. Set WRITE_TEMPLATE_CSV=True only if the group wants a scaffold CSV.")

print("Do not write the final file until real GraphRAG/PEFT evaluation has been run:")
print(D3_OUTPUT)

## 8. Actual D3 experiment that Aaya must run

When Rana's GraphRAG executor and Alia's evaluator are available, run the real comparison like this:

```text
same D3 gold Q/A stream
-> static GraphRAG retrieval
-> topic-gated GraphRAG retrieval
-> adaptive GraphRAG retrieval
-> optional adaptive GraphRAG with PEFT/QLoRA tuned model
-> compare retrieval metrics + RAG answer metrics + latency
```

Required comparison:

```text
static vs adaptive
```

Useful extra comparison:

```text
static vs topic_gated vs adaptive vs adaptive + PEFT/QLoRA
```

In [ ]:
# 9. Real implementation placeholder - keep runnable until GraphRAG executor exists.
# Replace these comments with actual imports once src/rag/graphrag_executor.py is implemented.

# from src.learning.river_topic_classifier import RiverTopicClassifier
# from src.learning.feedback_adapter import FeedbackAdapter
# from src.rag.graphrag_executor import GraphRAGExecutor
# from src.evaluation.rag_metrics import score_faithfulness, score_answer_relevance
# from src.safety.citation_verifier import verify_answer_citations

print("Next implementation step:")
print("1. Import RiverTopicClassifier and FeedbackAdapter.")
print("2. Import GraphRAGExecutor once Rana's D3 executor exists.")
print("3. Run static/topic_gated/adaptive on the same D3 gold Q/A rows.")
print("4. Save the real result to reports/tables/d3_online_retrieval_comparison.csv.")

## 10. Deep evaluation questions for Aaya

Use these questions while implementing and in the report:

1. Does adaptive routing improve the final answer, or only retrieval metrics?
2. Does the online learner help more during drift than during stable phases?
3. Does wrong topic prediction hurt GraphRAG by choosing the wrong graph path or retrieval weight?
4. Does PEFT/QLoRA tuning improve answer faithfulness, or only make answers sound better?
5. Is the extra complexity of adaptive routing justified by the quality/latency trade-off?
6. What cases should be reported as failures, not hidden?

## 11. Honest interpretation template

After the real D3 run, fill this in with actual numbers:

> Adaptive GraphRAG improved [metric] from [static value] to [adaptive value], mainly during [phase/query type]. It hurt [metric] in [case], usually when [wrong topic/noisy feedback/broad graph expansion]. PEFT/QLoRA [helped/did not help/could not be run] because [evidence]. The honest conclusion is [strong improvement / modest improvement / no reliable improvement], and the main limitation is [topic-level feedback / small gold set / graph coverage / hardware limit].